# Job Buddy

Upload a resume, say what you're looking for, and get Singapore jobs ranked against your profile.

**Run the cells in order.** Cell 1 sets up, cell 2 is the form, cell 3 checks your inputs, cell 4 searches.

Nothing here costs money — this stage uses only free public APIs and no LLM.

---

### What's compulsory

| Field | Why |
|---|---|
| **Resume** | Everything else is derived from it |
| **Your name** | Guessed from the resume, but you confirm it — a wrong name on a resume is unrecoverable |
| **Target roles** | The search terms |

Everything else is optional. Contact details, years of experience, skills and seniority are read out of the resume; anything you type wins over anything derived.

**Worth filling in anyway:** your current pay. Without it, pay scoring uses a generic market anchor instead of your actual position, and "is this a raise?" becomes guesswork.

In [ ]:
# --- Cell 1: setup -----------------------------------------------------
import importlib, sys
from pathlib import Path

import user_input as ui
importlib.reload(ui)

HAVE_WIDGETS = importlib.util.find_spec("ipywidgets") is not None

print(f"python      {sys.version.split()[0]}")
print(f"widgets     {'yes' if HAVE_WIDGETS else 'NO -- use the plain-text form in cell 2b'}")
print(f"pdf reader  {'yes' if importlib.util.find_spec('pypdf') else 'NO -- run: py -m pip install pypdf'}")
print(f"intake log  {ui.INTAKE_DIR}")

previous = ui.load_current()
if previous:
    print(f"\nfound a saved profile for {previous.get('full_name')!r} "
          f"(submitted {previous.get('submitted_at')})")
    print("the form below will prefill from it")

history = ui.submission_history(limit=5)
if history:
    print(f"\n{len(history)} recent submission(s):")
    for row in history:
        p = row["profile"]
        print(f"  {row['submitted_at']}  {p.get('full_name'):22} "
              f"{', '.join(p.get('target_roles') or [])[:40]:40} "
              f"resume={row.get('resume_archived_as')}")

## Cell 2 — the form

Drop your resume in the upload box (PDF, DOCX, TXT or MD), or paste a path to it.

Then press **Read resume & derive** to fill in everything it can, review what it found, and press **Save**.

In [ ]:
# --- Cell 2: interactive form (needs ipywidgets; if missing, use cell 2b)
if not HAVE_WIDGETS:
    print("ipywidgets not installed. Either run\n"
          "    py -m pip install ipywidgets\n"
          "and restart the kernel, or skip to cell 2b.")
else:
    import ipywidgets as W
    from IPython.display import display, clear_output

    prev = ui.load_current() or {}
    LABEL = {"description_width": "170px"}
    WIDE = W.Layout(width="620px")

    w_upload = W.FileUpload(accept=".pdf,.docx,.txt,.md", multiple=False,
                            description="Upload resume", layout=W.Layout(width="300px"))
    w_path = W.Text(value=prev.get("resume_path", ""), placeholder="...or paste a path",
                    description="Resume path *", style=LABEL, layout=WIDE)
    w_name = W.Text(value=prev.get("full_name", ""), description="Full name *",
                    style=LABEL, layout=WIDE)
    w_roles = W.Textarea(value=", ".join(prev.get("target_roles") or []),
                         placeholder="AI Engineer, Machine Learning Engineer, Data Scientist",
                         description="Target roles *", style=LABEL,
                         layout=W.Layout(width="620px", height="60px"))

    w_years = W.FloatText(value=prev.get("years_experience") or 0,
                          description="Years experience", style=LABEL)
    w_seniority = W.Dropdown(
        options=[("(derive from resume)", ""), ("junior", "junior"), ("mid", "mid"),
                 ("senior", "senior"), ("lead", "lead"), ("principal", "principal"),
                 ("manager", "manager"), ("director", "director")],
        value=prev.get("target_seniority") or "",
        description="Target seniority", style=LABEL)

    w_current_pay = W.IntText(value=prev.get("current_salary_sgd_monthly") or 0,
                              description="Current pay SGD/mo", style=LABEL)
    w_desired_pay = W.IntText(value=prev.get("desired_salary_sgd_monthly") or 0,
                              description="Desired pay SGD/mo", style=LABEL)
    w_min_pay = W.IntText(value=prev.get("min_salary_sgd_monthly") or 0,
                          description="Won't go below", style=LABEL)

    w_location = W.Text(value=prev.get("location", "Singapore"),
                        description="Location", style=LABEL, layout=WIDE)
    w_remote = W.Checkbox(value=prev.get("open_to_remote", True),
                          description="Open to remote", indent=False)
    w_authz = W.Text(value=prev.get("work_authorization", "Singapore Citizen / PR"),
                     description="Work authorisation", style=LABEL, layout=WIDE)
    w_notice = W.IntText(value=prev.get("notice_period_weeks") or 0,
                         description="Notice (weeks)", style=LABEL)

    w_exclude = W.Text(value=", ".join(prev.get("exclude_companies") or []),
                       placeholder="your current employer, and anyone you'd rather not hear from",
                       description="Exclude companies", style=LABEL, layout=WIDE)
    w_no_agencies = W.Checkbox(value=prev.get("exclude_agencies", False),
                               description="Hide recruitment agencies", indent=False)
    w_notes = W.Textarea(value=prev.get("notes", ""), placeholder="anything else worth recording",
                         description="Notes", style=LABEL,
                         layout=W.Layout(width="620px", height="60px"))

    b_derive = W.Button(description="Read resume & derive", button_style="info",
                        icon="search", layout=W.Layout(width="200px"))
    b_save = W.Button(description="Save profile", button_style="success",
                      icon="check", layout=W.Layout(width="200px"))
    out = W.Output()

    PROFILE = {"value": None}

    def _resolve_path():
        """Uploaded bytes win over the typed path; write them somewhere real."""
        if w_upload.value:
            item = (w_upload.value[0] if isinstance(w_upload.value, (list, tuple))
                    else list(w_upload.value.values())[0])
            name = item.get("name") or getattr(item, "name", "resume")
            content = item.get("content") if isinstance(item, dict) else item.content
            target = ui.INTAKE_DIR / "uploads" / name
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(bytes(content))
            return str(target)
        return w_path.value.strip()

    def _collect(path):
        overrides = {}
        if w_years.value:        overrides["years_experience"] = float(w_years.value)
        if w_seniority.value:    overrides["target_seniority"] = w_seniority.value
        if w_current_pay.value:  overrides["current_salary_sgd_monthly"] = int(w_current_pay.value)
        if w_desired_pay.value:  overrides["desired_salary_sgd_monthly"] = int(w_desired_pay.value)
        if w_min_pay.value:      overrides["min_salary_sgd_monthly"] = int(w_min_pay.value)
        if w_notice.value:       overrides["notice_period_weeks"] = int(w_notice.value)
        overrides["location"] = w_location.value.strip() or "Singapore"
        overrides["open_to_remote"] = bool(w_remote.value)
        overrides["work_authorization"] = w_authz.value.strip()
        overrides["exclude_agencies"] = bool(w_no_agencies.value)
        overrides["notes"] = w_notes.value.strip()
        if w_exclude.value.strip():
            overrides["exclude_companies"] = [c.strip() for c in w_exclude.value.split(",") if c.strip()]
        return ui.build_profile(full_name=w_name.value, resume_path=path,
                                target_roles=w_roles.value, **overrides)

    def on_derive(_):
        with out:
            clear_output()
            path = _resolve_path()
            if not path:
                print("Upload a resume, or paste a path to one.")
                return
            w_path.value = path
            profile = _collect(path)
            PROFILE["value"] = profile

            if not profile.resume_text:
                print(f"Could not read the resume: {profile.derived.get('resume_error')}")
                return

            # Reflect derived values back into the form so they are editable.
            if profile.full_name and not w_name.value:
                w_name.value = profile.full_name
            if profile.years_experience and not w_years.value:
                w_years.value = profile.years_experience
            if profile.target_seniority and not w_seniority.value:
                w_seniority.value = profile.target_seniority

            print(f"Read {len(profile.resume_text):,} characters "
                  f"via {profile.derived.get('resume_read')}\n")
            print("Derived from your resume")
            print("-" * 62)
            for label, value in (("name", profile.full_name), ("email", profile.email),
                                 ("phone", profile.phone), ("linkedin", profile.linkedin),
                                 ("years experience", profile.years_experience),
                                 ("current seniority", profile.current_seniority)):
                print(f"  {label:20} {value if value is not None else '-'}")
            print(f"  {'skills':20} {len(profile.skills)} found")
            if profile.skills:
                print(f"  {'':20} {', '.join(profile.skills[:12])}"
                      f"{' ...' if len(profile.skills) > 12 else ''}")
            basis = profile.derived.get("current_seniority")
            if basis:
                print(f"\n  seniority basis: {basis}")
            evidence = profile.derived.get("years_experience", {})
            if evidence.get("spans_found"):
                print(f"  dates used:      {'; '.join(evidence['spans_found'][:6])}")

            notes = ui.warnings_for(profile)
            if notes:
                print("\nWorth a look")
                print("-" * 62)
                for note in notes:
                    print(f"  - {note}")
            print("\nEdit anything above in the form, then press Save.")

    def on_save(_):
        with out:
            clear_output()
            path = _resolve_path()
            profile = _collect(path)
            problems = ui.validate(profile)
            if problems:
                print("Not saved. Fix these first:")
                for problem in problems:
                    print(f"  - {problem}")
                return
            PROFILE["value"] = profile
            submission_id = ui.log_submission(profile)
            print(f"Saved. Submission {submission_id}")
            print(f"  profile  {ui.CURRENT_PROFILE}")
            print(f"  log      {ui.SUBMISSIONS_LOG}")
            print(f"  resume   archived under {ui.RESUME_STORE}")
            for note in ui.warnings_for(profile):
                print(f"  note: {note}")
            print("\nRun cell 4 to search.")

    b_derive.on_click(on_derive)
    b_save.on_click(on_save)

    display(W.VBox([
        W.HTML("<h3>Required</h3>"),
        W.HBox([w_upload, W.HTML("<i style='margin-left:12px'>or paste a path below</i>")]),
        w_path, w_name, w_roles,
        W.HTML("<h3>Optional — derived from the resume if you leave them blank</h3>"),
        w_years, w_seniority,
        W.HTML("<h3>Pay <i>(strongly recommended — otherwise scoring uses a generic anchor)</i></h3>"),
        w_current_pay, w_desired_pay, w_min_pay,
        W.HTML("<h3>Preferences</h3>"),
        w_location, w_authz, w_notice, w_remote,
        w_exclude, w_no_agencies, w_notes,
        W.HTML("<br>"), W.HBox([b_derive, b_save]), out,
    ]))

## Cell 2b — plain-text form

Only needed if the widgets above didn't render. Edit the values, run the cell.

In [ ]:
# --- Cell 2b: no-widget fallback ---------------------------------------
# Edit these, run the cell. Leave a value as None to have it derived.

FULL_NAME    = ""                       # REQUIRED
RESUME_PATH  = "input/your_resume.pdf"  # REQUIRED
TARGET_ROLES = "AI Engineer, Machine Learning Engineer"   # REQUIRED

CURRENT_PAY_SGD_MONTHLY = None
DESIRED_PAY_SGD_MONTHLY = None
MIN_PAY_SGD_MONTHLY     = None
YEARS_EXPERIENCE        = None
TARGET_SENIORITY        = None    # junior|mid|senior|lead|principal|manager|director
EXCLUDE_COMPANIES       = []      # your current employer belongs here
EXCLUDE_AGENCIES        = False
NOTES                   = ""

# ------------------------------------------------------------------
_overrides = {k: v for k, v in {
    "current_salary_sgd_monthly": CURRENT_PAY_SGD_MONTHLY,
    "desired_salary_sgd_monthly": DESIRED_PAY_SGD_MONTHLY,
    "min_salary_sgd_monthly": MIN_PAY_SGD_MONTHLY,
    "years_experience": YEARS_EXPERIENCE,
    "target_seniority": TARGET_SENIORITY,
    "exclude_companies": EXCLUDE_COMPANIES,
    "exclude_agencies": EXCLUDE_AGENCIES,
    "notes": NOTES,
}.items() if v not in (None, "", [])}

profile = ui.build_profile(full_name=FULL_NAME, resume_path=RESUME_PATH,
                           target_roles=TARGET_ROLES, **_overrides)

problems = ui.validate(profile)
if problems:
    print("Not saved. Fix these first:")
    for p in problems:
        print(f"  - {p}")
else:
    print(f"name              {profile.full_name}")
    print(f"email             {profile.email}")
    print(f"years experience  {profile.years_experience}")
    print(f"seniority         {profile.target_seniority}")
    print(f"skills            {len(profile.skills)} found")
    sid = ui.log_submission(profile)
    print(f"\nSaved. Submission {sid}")
    for note in ui.warnings_for(profile):
        print(f"  note: {note}")

## Cell 3 — check what will actually be searched

Shows the config your profile produces, before anything runs. Worth a glance — the salary floor in particular silently removes a lot of jobs.

In [ ]:
# --- Cell 3: review the search config ----------------------------------
import json
import scoring
importlib.reload(scoring)

saved = ui.load_current()
if not saved:
    print("No saved profile. Run cell 2 (or 2b) and press Save first.")
else:
    p = ui.IntakeProfile(**{k: v for k, v in saved.items()
                            if k in ui.IntakeProfile.__dataclass_fields__})
    base = scoring.load_config()
    cfg = ui.to_run_config(p, base)

    print(f"Searching as   {p.full_name}")
    print(f"Roles          {', '.join(p.target_roles)}")
    print(f"Seniority      {cfg['profile'].get('target_seniority')}")
    print(f"Experience     {cfg['profile'].get('years_experience')} years")
    print()
    print("Hard filters (applied before scoring, and before anything costs money)")
    for k, v in cfg["filters"].items():
        if not k.startswith("_"):
            print(f"  {k:28} {v}")
    floor = cfg["filters"].get("min_salary_sgd_monthly")
    if floor:
        print(f"\n  NOTE: jobs whose TOP-of-range is under SGD {floor:,}/month are")
        print("        excluded outright. Lower it if too few results come back.")
    print()
    tiers = cfg["profile"].get("skills") or {}
    for tier in ("expert", "working", "familiar"):
        items = tiers.get(tier) or []
        if items:
            print(f"  {tier:9} ({len(items):2}) {', '.join(items[:10])}"
                  f"{' ...' if len(items) > 10 else ''}")
    print("\n  Skills from a resume land in 'working'. Promote the ones you could")
    print("  defend in an interview to 'expert' in run_config.json — the tier is a")
    print("  judgement only you can make, and it weights every match.")

## Cell 4 — search and rank

Hits MyCareersFuture (free, no key). Every Singapore employer is legally required to post there, so it reaches companies with no public careers API.

Results are written to `potential applications/` and your observation history to `state/`. Run it again in a few days and it will tell you what's new, what's been reposted, and how fast applications are arriving.

In [ ]:
# --- Cell 4: run the search --------------------------------------------
import slice1, job_store, source_mcf
for m in (slice1, job_store, source_mcf, scoring):
    importlib.reload(m)

MAX_PER_QUERY = 40      # lower this while experimenting

saved = ui.load_current()
if not saved:
    print("No saved profile. Run cell 2 (or 2b) and press Save first.")
else:
    p = ui.IntakeProfile(**{k: v for k, v in saved.items()
                            if k in ui.IntakeProfile.__dataclass_fields__})
    cfg = ui.to_run_config(p, scoring.load_config())
    scope = cfg["scopes"][0]
    scope["max_results_per_query"] = MAX_PER_QUERY

    run_id = job_store.new_run_id()
    prior = job_store.fold(job_store.read_sightings())
    velocity = job_store.company_velocity(prior)

    jobs, counters, excluded = slice1.collect(scope, cfg, None, cache_ttl_s=900.0)
    print(f"fetch: {counters}")

    if not jobs:
        print("\nNothing survived filtering. Most likely the salary floor is too high "
              "or the role names are too narrow — check cell 3.")
    else:
        job_store.record_sightings(jobs, run_id)
        history = job_store.fold(job_store.read_sightings())
        job_store.apply_history(jobs, history, run_id,
                                job_store.all_run_ids(job_store.read_sightings()))
        for job in jobs:
            scoring.score_job(job, cfg, velocity)
        jobs.sort(key=lambda j: j["scores"]["total"], reverse=True)

        new = sum(1 for j in jobs if j["job_key"] not in prior)
        print(f"\n{len(jobs)} ranked  ({new} new since last run)\n")
        for i, j in enumerate(jobs[:20], 1):
            pay = (f"{j['salary_min_sgd']}-{j['salary_max_sgd']}"
                   if j["salary_is_stated"] else "not stated")
            flag = "*" if j["job_key"] not in prior else " "
            print(f"{i:3d}{flag}{j['scores']['total']:5.0f}  {j['title'][:44]:44} "
                  f"{j['company'][:22]:22} {pay:13} apps={str(j['applications'] or '-'):>4}")

        out_path = (slice1.OUTPUT_DIR / scope["name"] / run_id / "ranked.csv")
        slice1.write_csv(jobs, out_path)
        job_store.write_snapshot(history, run_id)
        print(f"\nwrote {out_path}")
        if excluded:
            print(f"\n{len(excluded)} excluded by filters:")
            reasons = {}
            for e in excluded:
                reasons[e["reason"]] = reasons.get(e["reason"], 0) + 1
            for reason, n in sorted(reasons.items(), key=lambda x: -x[1])[:6]:
                print(f"  {n:4d}  {reason}")

## Cell 5 — dig into one job

Paste a `job_key` from the results to see every score input and its full history.

In [ ]:
# --- Cell 5: explain one job -------------------------------------------
JOB_KEY = ""     # e.g. "mcf:59501ac0a69b6c5cc91d7b8e9a23d276"

if not JOB_KEY:
    print("Set JOB_KEY to a job_key from the table above.")
else:
    rows = [r for r in job_store.read_sightings() if r.get("job_key") == JOB_KEY]
    if not rows:
        print(f"No history for {JOB_KEY}")
    else:
        print(f"=== {JOB_KEY} ===")
        print(f"seen {len(rows)} time(s)\n")
        print(f"{'when':22} {'apps':>5} {'views':>6} {'open':>6}")
        for r in rows:
            print(f"{r.get('ts',''):22} {str(r.get('applications')):>5} "
                  f"{str(r.get('views')):>6} {str(r.get('is_open')):>6}")
        if len(rows) > 1:
            first, last = rows[0].get("applications"), rows[-1].get("applications")
            if isinstance(first, int) and isinstance(last, int):
                print(f"\napplications while watching: {first} -> {last} "
                      f"({last - first:+d})")
        print("\nlatest record")
        for k in sorted(rows[-1]):
            print(f"  {k:22} {rows[-1][k]}")

---

## What gets logged, and where

| Path | Contents |
|---|---|
| `intake/current_profile.json` | The active profile the pipeline reads |
| `intake/submissions.jsonl` | Every submission, append-only — what you asked for and when |
| `intake/resumes/` | Every resume version, keyed by content hash |
| `intake/uploads/` | Raw uploads from the widget |
| `state/sightings.jsonl` | Every job seen, every run — this is what makes "new since last time" and application-velocity possible |
| `potential applications/` | Ranked output per run |

**All of these are gitignored.** They hold your name, phone number, salary expectations and job-search history, and this repo is public. The resume archive is content-hashed, so re-running with an unchanged resume doesn't duplicate it, and a past application can be reproduced with the exact document it was generated from.

### Known limits at this stage

- Derivation is regex, not an LLM. It's free and its mistakes are obvious rather than plausible — but **check the years-of-experience and seniority it reports**, since both feed the ranking directly.
- Skills land in the `working` tier. Promote what you'd defend in an interview to `expert` in `run_config.json`.
- MyCareersFuture is the only source wired up so far. It's the highest-coverage one for Singapore; the ATS adapters come next.
- No resume tailoring yet. That comes after `fact_guard.py`, which is what stops a generated resume claiming things you can't back up.